In [ ]:
import numpy as np

import seaborn as sns
sns.set_theme()

from helper_functions import (sphere_idx, write_text, interpolated_intercepts)

import scipy.constants as constants
import scipy.fft as fft

from matplotlib.ticker import FormatStrFormatter
from skimage.morphology import convex_hull_image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import h5py
import spimage
import glob
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

saveFigs = True
multiPRTF = False

<h2> Loading and calculating PRTF </h2> 
The 450 best images are selected for alignment/averaging and for the PRTF. 
For libspimage phase retrievals we are looking at the final real space error and select the best subset from this superset. 

In [ ]:
recon_directory = f'phasing/data_100k_prot_only_3_v_1/'
N_best = 500
noSupport = True
if noSupport:
    write_text('Loading non-support thresholded data!\n')
    recon_real = np.squeeze(np.load(recon_directory+'recon_real_nsp.npy'))
else:
    write_text('Loading support thresholded data!\n')
    recon_real = np.squeeze(np.load(recon_directory+'recon_real.npy'))
support_arr = np.squeeze(np.load(recon_directory+'support.npy'))
n_init = support_arr.shape[0]

real_error = np.load(recon_directory+'error_real.npy')
fourier_error = np.load(recon_directory+'error_fourier.npy')
sorted_real_final = np.argsort(real_error[:, -1])[:N_best]

prtf_res = spimage.prtf(recon_real[sorted_real_final, -1],support_arr[sorted_real_final,-1],full_out=True)
    
prtf_im = prtf_res['prtf']
prtf_im_abs = np.abs(prtf_im)

s_image = prtf_res['super_image']
s_image_abs = np.abs(s_image)

a_imgs = prtf_res['images']
a_msks = prtf_res['masks']

dimX, dimY, dimZ = a_imgs.shape[1:]
n_rec = a_imgs.shape[0]
center = dimX//2
write_text(f'There are {n_rec} reconstructions\n')
write_text(f'Removed {n_init-n_rec} reconstructions\n')

sel_rng = np.random.default_rng()
random_sel = np.squeeze(sel_rng.integers(0, N_best, size=1))
write_text(f'Selected random support: {random_sel}\n')
random_supp = a_msks[random_sel].copy()

if noSupport:
    out_directory = f'figures/'+recon_directory.split(sep='/')[1]+f'_{n_rec}_recs'+'_'+'nsp'+'_'
else:
    out_directory = f'figures/'+recon_directory.split(sep='/')[1]+f'_{n_rec}_recs'+'_'

<h2> Setting physical constants </h2>
This is particularly important to obtain the correct resolution for example, since the voxels themselves are dimensionless.
Some parameters are set by the simulations, others are different. For example, after EMC there are more pixels, so 
in principle we have higher resolution, but we won't have any signal there, so in practice this increase is purely a theoretical concept. This increase in array size is purely because of the need to orient the 2D diffraction patterns. Since the pattern might reach further than the array bounds set by simulations/experiments, the array size is increased in order to account for the 
The maximum resolution, or corner resolution is set by the simulation and the PRTF should not reach it.
The below voxel sizes are set by the dimensions of the EMC model, but have otherwise the same parameters as the simulation.

In [ ]:
e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 0.5
s_pixel = 2400e-6
edge_pixel = 45

D_particle = 15e-9

pixel_num = dimX - dimX//2
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector) 
resolution = lambda_photon/(2.0*np.sin(theta_pixel)) 
pix_real = 0.5 * resolution
voxel_size = pix_real

write_text(f'X-ray wavelength: {lambda_photon*1e9} nm\n')
write_text(f'Voxel size (real-space): {voxel_size*1e9} nm\n')
write_text(f'Voxel size (real-space): {voxel_size*1e10} Å\n')

theta_edge = 0.5 * np.arctan((edge_pixel*s_pixel)/d_detector) 
edge_res = lambda_photon/(2.0*np.sin(theta_edge))*1e9
edge_res_inv = 1 / edge_res

write_text(f'Edge resolution: {edge_res} nm\n')
write_text(f'Edge resolution: {edge_res*1e1} Å\n')

center_to_corner = np.sqrt((edge_pixel*s_pixel)**2+(edge_pixel*s_pixel)**2)
theta_corner = 0.5 * np.arctan((center_to_corner)/d_detector)
corner_res = lambda_photon/(2.0 * np.sin(theta_corner))*1e9
corner_res_inv = 1/corner_res

write_text(f'Corner resolution: {corner_res} nm\n')
write_text(f'Corner resolution: {corner_res*1e1} Å\n')

sh_pix_groel = 1 / (D_particle * 1e9) # in nm^-1
pix_emc = 1 / (dimX * voxel_size * 1e9) # in nm^-1
oversampling = sh_pix_groel / pix_emc

write_text(f'Shannon voxel GroEL: {sh_pix_groel} nm^-1\n') 
write_text(f'Voxel size Fourier: {pix_emc} nm^-1\n')
write_text(f'Fourier space oversampling: {oversampling}')

<h2> Plotting filtered Fourier/Real error </h2>
We will use the final real space error to do filtering on the reconstructions. 

In [ ]:
it_vec = np.linspace(0, 950, 100)
filt = sorted_real_final

fig_handle = plt.figure(1,constrained_layout = True, dpi = 150) 
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 2) 

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
plt.plot(it_vec, fourier_error[filt].T,'o-', ms=3)
plt.yscale('log')
ax_0.set_xticks([0, 500, 950])
ax_0.set_xlabel('iteration #', weight='bold')
ax_0.set_ylabel('error', weight='bold')
ax_0.set_title('Fourier space errors',weight='bold')

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
plt.plot(it_vec, real_error[filt].T,'o-', ms=3)
plt.yscale('log')
ax_1.set_xticks([0, 500, 950])
ax_1.set_xlabel('iteration #', weight='bold')
ax_1.set_ylabel('error', weight='bold')
ax_1.set_title('Real space errors',weight='bold');

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_reconstruction_errors_supp_{random_sel}.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);

<h2> Thresholding merged reconstruction </h2> 
This is to obtain the "merged support", since the "super_mask" returned by the Python interface of libspimage isn't
the correct "merged support". The most reliable way is to threshold the amplitude of the merged electron density of GroEL. It seems that for some reconstructions there is quite some sensitivity to the different threshold numbers in how many voxels we keep - for the resolution it doesn't seeem to matter a lot. There is slight shifts when changing the threshold. 

In [ ]:
sup_thresh = 0.15
s_support = s_image_abs > (sup_thresh * s_image_abs.max())
write_text(f'Number of points in thresholded support: {s_support.sum()}/{np.prod(s_support.shape)}\n')
write_text(f'Fraction of points in thresholded support: {s_support.mean()}\n')
 
showSupportMerged = True
if showSupportMerged:
    fig_handle = plt.figure(2,constrained_layout = True, dpi = 170)
    fig_handle.patch.set_facecolor(f'white')  
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3) 

    zoom = 40
    im_slice = center
    
    zy_diff = s_support[center-zoom:center+zoom,center-zoom:center+zoom,im_slice]
    zx_diff = s_support[center-zoom:center+zoom,im_slice,center-zoom:center+zoom]
    yx_diff = s_support[im_slice,center-zoom:center+zoom,center-zoom:center+zoom]

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(zy_diff,vmin=0,vmax=1,cmap='gray')
    ax_0.set_xticks([]) 
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim() 
    ax_0.set_title(f'ZY ({im_slice}/{dimX})',weight='bold',fontsize=6) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1])
    im_1 = plt.imshow(zx_diff,vmin=0,vmax=1,cmap='gray') 
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim()
    ax_1.set_title(f'ZX ({im_slice}/{dimY})',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yx_diff,vmin=0,vmax=1,cmap='gray') 
    ax_2.set_xticks([])
    ax_2.set_yticks([]) 
    minv, maxv = im_2.get_clim() 
    ax_2.set_title(f'YZ ({im_slice}/{dimZ})',weight='bold',fontsize=6);
    
if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{im_slice}_averaged_support_sup_{sup_thresh}.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0); 

In [ ]:
### Axis 0
max_num = 16
zoom = 40

im_slice = center

num_vox = a_msks.sum(axis=(1,2,3))
write_text(f'Number of voxels in individual reconstructions:\n {num_vox}')
def_rng = np.random.default_rng()
rand_sel = def_rng.choice(N_best,size=max_num,replace=False,shuffle=True)

fig_handle = plt.figure(constrained_layout = True,dpi=140)
fig_handle.patch.set_facecolor('white')
spec_handle = fig_handle.add_gridspec(nrows=4, ncols=4)
spec_handle.update(hspace=0.1,wspace=0.6)

for i in range(max_num): 
    sel = rand_sel[i]

    ax_i = fig_handle.add_subplot(spec_handle[i])
    im_i = plt.imshow(a_msks[sel][im_slice,center-zoom:center+zoom,center-zoom:center+zoom],
                      vmin=0.0,vmax=1.0,cmap='gray',interpolation=None)
    plt.suptitle(f'Axis 0', weight='bold',fontsize=10)
    ax_i.set_xticks([])
    ax_i.set_yticks([]) 
    ax_i.set_title(f'{sel}',fontsize=8, weight='bold')
    ax_i.set_aspect('equal');
    
### Axis 1
fig_handle = plt.figure(constrained_layout = True,dpi=140)
fig_handle.patch.set_facecolor('white')
spec_handle = fig_handle.add_gridspec(nrows=4, ncols=4)
spec_handle.update(hspace=0.1,wspace=0.6)

for i in range(max_num):
    sel = rand_sel[i]

    ax_i = fig_handle.add_subplot(spec_handle[i])
    im_i = plt.imshow(a_msks[sel][center-zoom:center+zoom, im_slice, center-zoom:center+zoom],
                      vmin=0.0,vmax=1.0,cmap='gray',interpolation=None)
    plt.suptitle(f'Axis 1', weight='bold',fontsize=10)
    ax_i.set_xticks([])
    ax_i.set_yticks([]) 
    ax_i.set_title(f'{sel}',fontsize=8, weight='bold')
    ax_i.set_aspect('equal');
    
### Axis 2
fig_handle = plt.figure(constrained_layout = True,dpi=140)
fig_handle.patch.set_facecolor('white')
spec_handle = fig_handle.add_gridspec(nrows=4, ncols=4)
spec_handle.update(hspace=0.1,wspace=0.6)

for i in range(max_num): 
    sel = rand_sel[i]

    ax_i = fig_handle.add_subplot(spec_handle[i])
    im_i = plt.imshow(a_msks[sel][center-zoom:center+zoom, center-zoom:center+zoom, im_slice],
                      vmin=0.0,vmax=1.0,cmap='gray',interpolation=None)
    plt.suptitle(f'Axis 2', weight='bold',fontsize=10)
    ax_i.set_xticks([])
    ax_i.set_yticks([])
    ax_i.set_title(f'{sel}',fontsize=8, weight='bold')
    ax_i.set_aspect('equal');

<h2> Maximum-projections of merged density </h2>

In [ ]:
max_proj_0 = s_image_abs.max(axis=0)
max_proj_1 = s_image_abs.max(axis=1)
max_proj_2 = s_image_abs.max(axis=2)

c_proj = max_proj_0.shape[0]//2
c_max = s_image_abs.max(axis=(0,1,2))
c_zoom = 40

fig_handle = plt.figure(1,constrained_layout = True, dpi = 180)
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
cm = 'viridis'

ax_0 = fig_handle.add_subplot(spec_handle[0,0]) 
im_0 = plt.imshow(max_proj_0[c_proj-c_zoom:c_proj+c_zoom,c_proj-c_zoom:c_proj+c_zoom],vmin=0.,vmax=c_max,interpolation=None,cmap=cm)
plt.title('Axis 0', weight='bold')
ax_0.set_xticks([]) 
ax_0.set_yticks([]) 
minv, maxv = im_0.get_clim() 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.4, orientation='vertical') 
c_bar_0.set_ticks([minv, 0.5*c_max, c_max]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
im_1 = plt.imshow(max_proj_1[c_proj-c_zoom:c_proj+c_zoom,c_proj-c_zoom:c_proj+c_zoom],vmin=0.,vmax=c_max,interpolation=None,cmap=cm) 
plt.title('Axis 1', weight='bold')
ax_1.set_xticks([]) 
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
im_2 = plt.imshow(max_proj_2[c_proj-c_zoom:c_proj+c_zoom,c_proj-c_zoom:c_proj+c_zoom],vmin=0.,vmax=c_max,interpolation=None,cmap=cm)
plt.title('Axis 2', weight='bold')
ax_2.set_xticks([]) 
ax_2.set_yticks([])
plt.autoscale();

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_max_projections.pdf',format='pdf',dpi=120,bbox_inches='tight',pad_inches=0.0); 
    
if saveFigs: 
    rd = recon_directory.split(sep='/')
    if noSupport: # nsp for no support enforced on density
        np.save(f'average_reconstructions/'+rd[1]+f'_nsp_{n_rec}r', s_image_abs)
        np.save(f'average_reconstructions/'+rd[1]+'_nsp'+f'_support_{n_rec}r', s_support)
    else: # sp for support enforced on density
        np.save(f'average_reconstructions/'+rd[1]+f'_sp_{n_rec}r',s_image_abs)
        np.save(f'average_reconstructions/'+rd[1]+'_sp'+f'_support_{n_rec}r',s_support)

<h2> Mean-projections of merged density </h2>

In [ ]:
mean_proj_0 = s_image_abs.mean(axis=0)
mean_proj_1 = s_image_abs.mean(axis=1)
mean_proj_2 = s_image_abs.mean(axis=2)

c_proj = mean_proj_0.shape[0]//2
c_max = s_image_abs.mean(axis=(0,1,2))*40

c_zoom = 40

fig_handle = plt.figure(1,constrained_layout = True, dpi = 180)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
cm = 'viridis'

ax_0 = fig_handle.add_subplot(spec_handle[0,0]) 
im_0 = plt.imshow(mean_proj_0[c_proj-c_zoom:c_proj+c_zoom,c_proj-c_zoom:c_proj+c_zoom],vmin=0.,vmax=c_max,interpolation=None,cmap=cm)
plt.title('Axis 0', weight='bold')
ax_0.set_xticks([]) 
ax_0.set_yticks([]) 
minv, maxv = im_0.get_clim() 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.4, orientation='vertical') 
c_bar_0.set_ticks([minv, 0.5*c_max, c_max]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
im_1 = plt.imshow(mean_proj_1[c_proj-c_zoom:c_proj+c_zoom,c_proj-c_zoom:c_proj+c_zoom],vmin=0.,vmax=c_max,interpolation=None,cmap=cm) 
plt.title('Axis 1', weight='bold')
ax_1.set_xticks([]) 
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
im_2 = plt.imshow(mean_proj_1[c_proj-c_zoom:c_proj+c_zoom,c_proj-c_zoom:c_proj+c_zoom],vmin=0.,vmax=c_max,interpolation=None,cmap=cm)
plt.title('Axis 2', weight='bold')
ax_2.set_xticks([]) 
ax_2.set_yticks([])
plt.autoscale();

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_mean_projections.pdf',format='pdf',dpi=120,bbox_inches='tight',pad_inches=0.0);

<h2> Radial average of threshold electron density </h2> 

In [ ]:
dens_avg = spimage.radial(s_image_abs)
dens_avg /= dens_avg.max()

max_points = dens_avg.shape[0] 
vector = np.linspace(0, max_points, num=max_points) 
vector = vector * voxel_size * 1e9

plt.figure(dpi=140)
plt.plot(vector, dens_avg, 'k-')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('distance from center $(nm)$', weight='bold')
plt.ylabel('average density $(arb. unit)$', weight='bold')

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_radial_average_density.pdf',format='pdf', dpi=200, bbox_inches='tight', pad_inches=0.0);

<h2> Visualizing XY, XZ, and YZ slices of merged super-reconstruction/PRTF </h2>

In [ ]:
im_slice = dimX//2

fig_handle = plt.figure(1,constrained_layout = True, dpi = 220)
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 2, ncols = 3) 
plt.suptitle(f'Slices of {n_rec} merged reconstructions')
cm = 'coolwarm' 

# Reconstructed and merged electron density amplitude
max_v = 1.3 * s_image_abs.max()
zoom = 40

xy_obj = s_image_abs[center-zoom:center+zoom,center-zoom:center+zoom,im_slice] 
xz_obj = s_image_abs[center-zoom:center+zoom,im_slice,center-zoom:center+zoom] 
yz_obj = s_image_abs[im_slice,center-zoom:center+zoom,center-zoom:center+zoom] 

ax_0 = fig_handle.add_subplot(spec_handle[0,0]) 
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_0.set_xticks([]) 
ax_0.set_yticks([]) 
minv, maxv = im_0.get_clim() 
ax_0.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.01,shrink=0.6,orientation='horizontal') 
c_bar_0.set_ticks([minv,0.5*maxv,maxv]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_1.set_xticks([]) 
ax_1.set_yticks([]) 
minv, maxv = im_1.get_clim() 
ax_1.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_2.set_xticks([]) 
ax_2.set_yticks([]) 
minv, maxv = im_2.get_clim() 
ax_2.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6) 

# PRTF projections  
max_v = 1e0 

xy_obj = prtf_im_abs[:,:,im_slice]
xz_obj = prtf_im_abs[:,im_slice,:]
yz_obj = prtf_im_abs[im_slice,:,:]

ax_3 = fig_handle.add_subplot(spec_handle[1,0]) 
im_3 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_3.set_xticks([]) 
ax_3.set_yticks([])
minv, maxv = im_3.get_clim()
ax_3.set_title(f'PRTF (XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_3 = plt.colorbar(im_3, ax=ax_3,fraction=0.01,shrink=0.6,orientation='horizontal') 
c_bar_3.set_ticks([minv,0.5*maxv,maxv]) 

ax_4 = fig_handle.add_subplot(spec_handle[1,1]) 
im_4 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_4.set_xticks([])
ax_4.set_yticks([])
minv, maxv = im_4.get_clim()
ax_4.set_title(f'PRTF (XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_5 = fig_handle.add_subplot(spec_handle[1,2]) 
im_5 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_5.set_xticks([]) 
ax_5.set_yticks([]) 
minv, maxv = im_5.get_clim() 
ax_5.set_title(f'PRTF (YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

if saveFigs: 
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{im_slice}_overview.pdf',format='pdf',dpi=220,bbox_inches='tight',pad_inches=0.0); 

<h2> Radial average (2D --> 1D) PRTF for all slices </h2> 
A random reconstruction support mask is used to convolve the PRTF with, to generate a critically-sampled PRTF. 

In [ ]:
sel_slice = center

prtf_x = spimage.radial(prtf_im[sel_slice])
prtf_y = spimage.radial(prtf_im[:,sel_slice,:])
prtf_z = spimage.radial(prtf_im[:,:,sel_slice])

conv_hull_supp_x = convex_hull_image(random_supp[sel_slice])
conv_hull_supp_y = convex_hull_image(random_supp[:,sel_slice,:])
conv_hull_supp_z = convex_hull_image(random_supp[:,:,sel_slice])

prtf_x_ft = fft.ifftn((prtf_im[sel_slice]))
prtf_y_ft = fft.ifftn(prtf_im[:,sel_slice,:])
prtf_z_ft = fft.ifftn((prtf_im[:,:,sel_slice]))

prtf_x_smooth = spimage.radial(np.abs(fft.fftn(prtf_x_ft * fft.fftshift(conv_hull_supp_x))))
prtf_y_smooth = spimage.radial(np.abs(fft.fftn(prtf_y_ft * fft.fftshift(conv_hull_supp_y))))
prtf_z_smooth = spimage.radial(np.abs(fft.fftn(prtf_z_ft * fft.fftshift(conv_hull_supp_z))))

prtf_x_smooth /= prtf_x_smooth.max()
prtf_y_smooth /= prtf_y_smooth.max()
prtf_z_smooth /= prtf_z_smooth.max()

plotIP = True 
threshold = 1/np.exp(1)

fig_handle = plt.figure(3,constrained_layout = True, dpi = 280) 
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 2, ncols = 9) 
marker_size = 3.0

### XY 
max_points = prtf_z.shape[0]
fp_resolution_inv = np.arange(0, max_points) * pix_emc # in nm^-1

xc1, yc1 = interpolated_intercepts(fp_resolution_inv, prtf_z,np.repeat(threshold, max_points)) 
xc11, yc11 = interpolated_intercepts(fp_resolution_inv, prtf_z_smooth,np.repeat(threshold, max_points)) 

write_text(f'-----XY/XY/YZ-----\n') 
write_text(f'Full period resolution cutoff: {1/fp_resolution_inv[-1]:.3f} nm\n') 

ax_0 = fig_handle.add_subplot(spec_handle[0,:3]) 
plt.plot(fp_resolution_inv,prtf_z, color=mcolors.XKCD_COLORS['xkcd:apple green'], linestyle='--')
plt.plot(fp_resolution_inv,prtf_z_smooth, color=mcolors.XKCD_COLORS['xkcd:jungle green']) 
ax_0.axhline(y=threshold,xmin=0, xmax=(fp_resolution_inv[-1])*max_points, c=mcolors.XKCD_COLORS['xkcd:light blue'], linestyle='-') 
ax_0.xaxis.set_major_formatter(FormatStrFormatter('%.3f')) 
ax_0.set_title(f'XY ({im_slice}/{dimX})',weight='bold',fontsize=6) 
ax_0.set_xlim([0, fp_resolution_inv[-1]+0.015]) 
ax_0.set_ylim([0, 1.05]) 
ax_0.set_xticks(fp_resolution_inv[::40]) 
ax_0.set_xlabel('|q| $(nm^{-1})$',weight='bold') 
ax_0.set_ylabel('PRTF',weight='bold') 
ax_0.legend(['PRTF','PRTF conv','1/e cutoff'],frameon = False,prop=dict(weight='bold',size=6),loc=0,fontsize=0.5) 

if plotIP: 
    if xc11.size != 0: 
        ax_0.plot(xc11,np.repeat(threshold,xc11.size),'ko',ms=marker_size)
        write_text(f'Intersection(s) smoothened PRTF with 1/e threshold: {1/xc11} nm\n')

### XZ
xc2, yc2 = interpolated_intercepts(fp_resolution_inv, prtf_y,np.repeat(threshold,max_points)) 
xc22, yc22 = interpolated_intercepts(fp_resolution_inv, prtf_y_smooth,np.repeat(threshold,max_points)) 

ax_1 = fig_handle.add_subplot(spec_handle[0,3:6]) 
im_1 = plt.plot(fp_resolution_inv,prtf_y, color=mcolors.XKCD_COLORS['xkcd:apple green'], linestyle='--')
plt.plot(fp_resolution_inv,prtf_y_smooth, color=mcolors.XKCD_COLORS['xkcd:jungle green']) 
ax_1.axhline(y=threshold, xmin=0, xmax=(fp_resolution_inv[-1])*max_points, c=mcolors.XKCD_COLORS['xkcd:light blue'], linestyle='-')
ax_1.xaxis.set_major_formatter(FormatStrFormatter('%.3f')) 
ax_1.set_title(f'XZ ({im_slice}/{dimX})',weight='bold',fontsize=6) 
ax_1.set_xlim([0,fp_resolution_inv[-1]+0.015])
ax_1.set_ylim([0,1.05]) 
ax_1.set_xticks(fp_resolution_inv[::40]) 
ax_1.set_xlabel('|q| $(nm^{-1})$',weight='bold')
ax_1.set_ylabel('PRTF',weight='bold')
ax_1.legend(['PRTF','PRTF conv','1/e cutoff'],frameon = False,prop=dict(weight='bold',size=6),loc=0,fontsize=0.5)
    
if plotIP:
    if xc22.size!= 0: 
        ax_1.plot(xc22,np.repeat(threshold,xc22.size),'ko',ms=marker_size) 
        write_text(f'Intersection(s) smoothened PRTF with 1/e threshold: {1/xc22} nm\n')
    
### YZ
xc3, yc3 = interpolated_intercepts(fp_resolution_inv, prtf_x,np.repeat(threshold,max_points)) 
xc33, yc33 = interpolated_intercepts(fp_resolution_inv, prtf_x_smooth,np.repeat(threshold,max_points)) 

ax_2 = fig_handle.add_subplot(spec_handle[0,6:]) 
im_2 = plt.plot(fp_resolution_inv,prtf_x, color=mcolors.XKCD_COLORS['xkcd:apple green'], linestyle='--') 
plt.plot(fp_resolution_inv,prtf_x_smooth, color=mcolors.XKCD_COLORS['xkcd:jungle green'])
ax_2.axhline(y=threshold, xmin=0, xmax=(fp_resolution_inv[-1])*max_points, c=mcolors.XKCD_COLORS['xkcd:light blue'], linestyle='-')
ax_2.xaxis.set_major_formatter(FormatStrFormatter('%.3f')) 
ax_2.set_title(f'YZ ({im_slice}/{dimX})',weight='bold',fontsize=6)
ax_2.set_xlim([0,fp_resolution_inv[-1]+0.015]) 
ax_2.set_ylim([0, 1.05]) 
ax_2.set_xticks(fp_resolution_inv[::40]) 
ax_2.set_xlabel('|q| $(nm^{-1})$',weight='bold') 
ax_2.set_ylabel('PRTF',weight='bold')
ax_2.legend(['PRTF','PRTF conv','1/e cutoff'],frameon = False,prop=dict(weight='bold',size=6),loc=0,fontsize=0.5)

if plotIP:
    if xc33.size != 0:
        ax_2.plot(xc33,np.repeat(threshold,xc33.size),'ko',ms=marker_size)
        write_text(f'Intersection(s) smoothened PRTF with 1/e threshold: {1/xc33} nm\n')
        
if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{sel_slice}_prtf_xyz.pdf',format='pdf',dpi=250,bbox_inches='tight',pad_inches=0.0); 

<h2> Radial average of PRTF (3D --> 1D) </h2> 
Before calculating the radial average, we will use a similar method that was used in the past whereby the 3D PRTF was blurred using
a top-hat kernel of size equal to object support. This leads to a critically-sampled PRTF that is subsquently radially-integrated to yield the 1D radial average. The modification in our current approach is that instead of top-hat kernel, we use a randomly selected support from the N best reconstructions. We also take the convex hull to get the shape only without any gaps. 

In [ ]:
conv_hull_supp = convex_hull_image(random_supp)
prtf_im_ft = fft.ifftn((prtf_im))
prtf_im_smooth = np.abs(fft.fftn(prtf_im_ft * fft.fftshift(conv_hull_supp)))

fig_handle = plt.figure(constrained_layout = True, dpi = 220)
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3) 
cm = 'coolwarm'
im_slice = center

xy_obj = prtf_im_smooth[:,:,im_slice]
xz_obj = prtf_im_smooth[:,im_slice,:]
yz_obj = prtf_im_smooth[im_slice,:,:]

ax_3 = fig_handle.add_subplot(spec_handle[0])
im_3 = plt.imshow(xy_obj,cmap=cm,interpolation=None)
ax_3.set_xticks([]) 
ax_3.set_yticks([])
minv, maxv = im_3.get_clim()
ax_3.set_title(f'PRTF (XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_3 = plt.colorbar(im_3,ax=ax_3,fraction=0.001,shrink=0.425,orientation='vertical') 
c_bar_3.set_ticks([minv,maxv,maxv])

ax_4 = fig_handle.add_subplot(spec_handle[1]) 
im_4 = plt.imshow(xz_obj,cmap=cm,interpolation=None) 
ax_4.set_xticks([])
ax_4.set_yticks([])
ax_4.set_title(f'PRTF (XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_5 = fig_handle.add_subplot(spec_handle[2]) 
im_5 = plt.imshow(yz_obj,cmap=cm,interpolation=None) 
ax_5.set_xticks([]) 
ax_5.set_yticks([])
ax_5.set_title(f'PRTF (YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

In [ ]:
rad_sh = 1 # radius denoting thickness of spherical shell # in voxels
rad_vox = pix_emc

prtf_r = spimage.radial(prtf_im, shell_thickness=rad_sh)

prtf_r_smooth = spimage.radial(prtf_im_smooth, shell_thickness=rad_sh)
prtf_r_smooth /= np.max(prtf_r_smooth)

max_points = prtf_r.shape[0] # maximum number of voxels from PRTF/PRTF_smooth

if rad_sh == 1:
    write_text(f'Size of spherical shell: {rad_sh} voxel(s) or {rad_vox} nm^-1\n')
else:
    rad_vox *= rad_sh
    write_text(f'Size of spherical shell: {rad_sh} voxel(s) or {rad_vox} nm^-1\n') 

# PRTF resolution vector
fp_resolution_r_inv = np.arange(0, max_points) * rad_vox # in nm^-1

xcp, ycp = interpolated_intercepts(fp_resolution_r_inv, prtf_r, np.repeat(threshold, max_points))
xcp_s, ycp_s = interpolated_intercepts(fp_resolution_r_inv, prtf_r_smooth, np.repeat(threshold, max_points))
    
plt.figure(4,dpi=150)
plt.plot(fp_resolution_r_inv, prtf_r, color=mcolors.XKCD_COLORS['xkcd:apple green'], linestyle='--')
plt.plot(fp_resolution_r_inv, prtf_r_smooth, color=mcolors.XKCD_COLORS['xkcd:jungle green'])

plt.axhline(y=threshold, xmin=0, xmax=1.0, c=mcolors.XKCD_COLORS['xkcd:light blue'], linestyle='-')
    
showLimits = True
if showLimits:
    plt.axvline(x=edge_res_inv, ymin=0 , ymax=1-0.01, c=mcolors.XKCD_COLORS['xkcd:red'],linestyle='--', alpha=0.4)
    plt.axvline(x=fp_resolution_r_inv[-1], ymin=0 , ymax=1-0.01, c=mcolors.XKCD_COLORS['xkcd:blue'],linestyle='--', alpha=0.4)
    plt.axvline(x=corner_res_inv, ymin=0 , ymax=1-0.01, c=mcolors.XKCD_COLORS['xkcd:green'],linestyle='--', alpha=0.4)
    plt.xlim([-0.002, corner_res_inv+0.005])
else:
    plt.xlim([-0.002, fp_resolution_r_inv[-1]+0.01])
    
plt.ylim([0.0, 1.01])
plt.xlabel('|q| $(nm^{-1})$', weight='bold')
plt.ylabel('PRTF', weight='bold')
plt.legend([f'PRTF ({n_rec})', f'PRTF conv ({n_rec})', '1/e cutoff'], frameon = False, prop=dict(weight='bold', size=8), fontsize=0.5, loc=0)

if plotIP:
    if (xcp.size != 0) or (xcp_s.size != 0):
        plt.plot(xcp_s, np.repeat(threshold, xcp_s.size),'ko', ms=4.0)
        write_text(f'Intersection(s) smoothened PRTF with 1/e threshold: {1/xcp_s} nm\n')
        
if saveFigs:
    with h5py.File(f'figures/'+recon_directory.split(sep='/')[1]+'_PRTF.h5') as prtf_handle:
        prtf_handle['prtf_r_smooth'] = prtf_r_smooth
        prtf_handle['prtf_r'] = prtf_r
        prtf_handle['prtf_3d'] = prtf_im_smooth
        prtf_handle['fp_res_inv_nm'] = fp_resolution_r_inv
        prtf_handle['res_r_smooth_nm'] = 1/xcp_s
        prtf_handle['res_r_nm'] = 1/xcp
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+'_PRTF.pdf', format='pdf', dpi=150, bbox_inches='tight', pad_inches=0.0);

<h2> Plotting multiple PRTFs in single figure </h2> 

In [ ]:
filesPRTF = np.sort(np.array(glob.glob(f'figures/*_v_2_*.h5')))
if multiPRTF:
    names_PRTF = []
    combs_PRTF = []
    res_PRTF = []
 
    for file in filesPRTF:
        f_name = file.split(sep='/')[1].split(sep='.h5')[0]
        with h5py.File(file) as f:
            combs_PRTF.append(f['prtf_r'][:])
            res_PRTF.append(f['fp_res_inv_nm'][:])
        names_PRTF.append(f_name)

    names_PRTF = np.array(names_PRTF)
    combs_PRTF = np.array(combs_PRTF)
    res_PRTF = np.array(res_PRTF)
    max_points = res_PRTF.shape[1]

    res_ax = res_PRTF[0]

    plt.figure(dpi=140)
    plt.axvline(x=0.7805673883895187, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_')
    plt.axhline(y=1/np.exp(1), xmin=0.0, xmax=res_ax[-1]*10, c='k', linestyle='--', linewidth=1.0, label='_nolegend_')
    for p in range(len(filesPRTF)):
        plt.plot(res_ax, combs_PRTF[p], linestyle='-')

    plt.xlim([0, res_ax[-1]])
    plt.ylim([-0.14, 1.035])
    plt.ylim([0.0, 1.01])
    plt.xlabel('|q| $(nm^{-1})$', weight='bold')
    plt.ylabel('PRTF', weight='bold')
    plt.legend(names_PRTF, frameon=True, prop=dict(size=9), loc=1);
    #plt.title('Prot-wat', weight='bold');